In [0]:
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import functions as F

CATALOG_SCHEMA = "teste_koin.default."
SILVER_ORDERS_TABLE    = "silver_orders"
SILVER_CUSTOMERS_TABLE = "silver_customers"
GOLD_TABLE             = "gold_orders_customers"

MERGE_KEY = "order_id"

# Logging simples
def log(message: str):
    print(f"[{datetime.now()}] {message}")

# Leitura das tabelas Silver
df_orders = spark.table(CATALOG_SCHEMA + SILVER_ORDERS_TABLE)
df_customers = spark.table(CATALOG_SCHEMA + SILVER_CUSTOMERS_TABLE)

# LEFT JOIN
df_gold = (
    df_orders.alias("o")
    .join(
        df_customers.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )

    .select(
        # Orders
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("o.order_amount"),
        F.col("o.order_payment_method"),
        F.col("o.order_status"),
        F.col("o.order_date"),
              
        # Customers
        F.col("c.customer_name"),
        F.col("c.customer_cpf_hash"),
        F.col("c.customer_email_hash"),
        F.col("c.customer_masked_email"),
        F.col("c.customer_masked_phone"),
        F.col("c.customer_city"),
        F.col("c.customer_state"),
        F.col("c.customer_created_at_date"),
        F.col("c.customer_status"),

        # Auditoria
        F.current_timestamp().alias("gold_at")
    )
)

# Escrita Gold
if spark.catalog.tableExists(CATALOG_SCHEMA + GOLD_TABLE):

    delta_table = DeltaTable.forName(
        spark,
        CATALOG_SCHEMA + GOLD_TABLE
    )

    (
        delta_table.alias("target")
        .merge(
            df_gold.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("MERGE Gold concluído com sucesso.")

else:

    (
        df_gold.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(CATALOG_SCHEMA + GOLD_TABLE)
    )

    print("Tabela Gold criada com sucesso.")